In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Layer 2 Intermediate Acuity: Hierarchical Binary Logistic Regressors (`models/lr_feng_esi234_extreme.ipynb`)

This notebook trains **2 individual Binary Logistic Regressors** for the intermediate ESI tiers (ESI 2, 3, 4) with **Configurable Target Class Subsampling**:

1. **Model 1: ESI 2 vs. Not ESI 2**:
   - **Dataset Exclusion**: **ESI 1 and ESI 5 rows are completely removed** prior to partitioning into Train, Validation, and Test splits.
   - **Target Output**: Predicts whether a patient is **ESI 2** (`"2"`) or **Not ESI 2** (`"not_2"`, which comprises ESI 3 and ESI 4).
   - **Configurable Subsampling**: Adjust `keep_ratio_2` and `keep_ratio_not_2` in Step 2A.
   - **Model Engine**: Binary Logistic Regressor saved to `deploy/lr_feng_esi2_model.rds`.

2. **Model 2: ESI 3 vs. ESI 4**:
   - **Dataset Exclusion**: **ESI 1, ESI 5, AND ESI 2 rows are completely removed** prior to partitioning into Train, Validation, and Test splits.
   - **Target Output**: Predicts whether a patient is **ESI 3** (`"3"`) or **ESI 4** (`"4"`).
   - **Configurable Subsampling**: Adjust `keep_ratio_3` and `keep_ratio_4` in Step 2B.
   - **Model Engine**: Binary Logistic Regressor saved to `deploy/lr_feng_esi34_model.rds`.

Both models use the **13 Feature Engineered Predictors** (`age`, `gender`, `cc_breathingdifficulty`, and 10 vital sign flags) and evaluate **Accuracy, Precision, Recall (Sensitivity), PR-AUC, and Log Loss** across splits.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(nnet)
library(dplyr)
library(ggplot2)
library(tidyr)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Shared Evaluation Helper Functions (PR-AUC & Log Loss)
# ---------------------------------------------------------
calc_pr_auc <- function(actual_binary, prob_positive) {
  tryCatch({
    ord <- order(prob_positive, decreasing = TRUE)
    act_sorted <- (actual_binary[ord] == 1)
    tp <- cumsum(act_sorted)
    fp <- cumsum(!act_sorted)
    n_pos <- sum(act_sorted)
    if (n_pos == 0) return(NA)
    rec <- c(0, tp / n_pos)
    prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
    dx <- diff(rec)
    my <- (prec[-1] + prec[-length(prec)]) / 2
    return(as.numeric(sum(dx * my)))
  }, error = function(e) NA)
}

calc_log_loss <- function(actual_factor, prob_matrix, eps = 1e-15) {
  if (!is.matrix(prob_matrix)) {
    prob_matrix <- cbind(1 - prob_matrix, prob_matrix)
    colnames(prob_matrix) <- levels(actual_factor)
  }
  prob_matrix <- pmax(pmin(prob_matrix, 1 - eps), eps)
  prob_matrix <- prob_matrix / rowSums(prob_matrix)
  classes <- colnames(prob_matrix)
  N <- length(actual_factor)
  
  log_probs <- numeric(N)
  for (i in 1:N) {
    act_cls <- as.character(actual_factor[i])
    if (act_cls %in% classes) {
      log_probs[i] <- log(prob_matrix[i, act_cls])
    } else {
      log_probs[i] <- log(eps)
    }
  }
  return(-mean(log_probs))
}

evaluate_binary_lr <- function(model, data, target_col_name, pos_class, model_name, set_name) {
  prob_res <- predict(model, newdata = data, type = "probs")
  target_classes <- levels(data[[target_col_name]])
  neg_class <- setdiff(target_classes, pos_class)[1]
  
  if (is.matrix(prob_res)) {
    prob_matrix <- prob_res
  } else {
    prob_matrix <- cbind(1 - prob_res, prob_res)
    colnames(prob_matrix) <- c(pos_class, neg_class)
  }
  
  max_idx <- max.col(prob_matrix, ties.method = "first")
  pred_factor <- factor(colnames(prob_matrix)[max_idx], levels = target_classes)
  actual_factor <- factor(data[[target_col_name]], levels = target_classes)
  
  cm <- confusionMatrix(pred_factor, actual_factor, positive = pos_class)
  acc  <- as.numeric(cm$overall["Accuracy"])
  prec <- as.numeric(cm$byClass["Pos Pred Value"])
  rec  <- as.numeric(cm$byClass["Sensitivity"])
  
  if (is.na(prec)) prec <- 0
  if (is.na(rec))  rec  <- 0
  
  prob_pos <- prob_matrix[, pos_class]
  act_binary <- ifelse(actual_factor == pos_class, 1, 0)
  pr_auc <- calc_pr_auc(act_binary, prob_pos)
  
  log_loss <- calc_log_loss(actual_factor, prob_matrix)
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   %s - %s SET BENCHMARK\n", toupper(model_name), toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Accuracy             : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  Precision            : %.4f (%.2f%%)\n", prec, prec * 100))
  cat(sprintf("  Recall (Sensitivity) : %.4f (%.2f%%)\n", rec, rec * 100))
  cat(sprintf("  PR-AUC               : %.4f\n", pr_auc))
  cat(sprintf("  Log Loss             : %.4f\n", log_loss))
  cat("\nTarget Class Counts (Actual vs Predicted Comparison):\n")
  class_counts_df <- data.frame(
    Class = target_classes,
    Actual_Count = as.numeric(table(actual_factor)[target_classes]),
    Predicted_Count = as.numeric(table(pred_factor)[target_classes]),
    Diff = as.numeric(table(pred_factor)[target_classes]) - as.numeric(table(actual_factor)[target_classes])
  )
  print(class_counts_df)
  cat("\nConfusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
  
  return(list(acc = acc, prec = prec, rec = rec, pr_auc = pr_auc, log_loss = log_loss, prob_matrix = prob_matrix, actual_factor = actual_factor))
}

--- 
## SECTION A: MODEL 1 - BINARY LOGISTIC REGRESSOR FOR ESI 2 (ESI 2 vs. Not ESI 2)
**Dataset Condition**: ESI 1 and ESI 5 rows are completely excluded (`raw_esi != "1" & raw_esi != "5"`).

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2A: Load Data, Exclude ESI 1 & 5, Construct Target & Apply Class Subsampling
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0

df_feng_m1 <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)

raw_esi_m1 <- as.character(raw_df[[target_col]])

# EXCLUDE ESI 1 AND ESI 5 COMPLETELY FOR MODEL 1
idx_m1 <- which(!(raw_esi_m1 %in% c("1", "5")) & !is.na(raw_esi_m1))
df_feng_m1  <- df_feng_m1[idx_m1, ]
raw_esi_sub1 <- raw_esi_m1[idx_m1]

# Target: '2' vs 'not_2'
df_feng_m1$target_esi2 <- factor(ifelse(raw_esi_sub1 == "2", "2", "not_2"), levels = c("2", "not_2"))

# ---------------------------------------------------------
# Configurable Subsampling Ratios for Model 1
# ---------------------------------------------------------
keep_ratio_2     <- 1.00  # Keep 100% of ESI 2 rows (configurable)
keep_ratio_not_2 <- 0.2  # Keep 100% of 'not_2' rows (configurable)

idx_2     <- which(df_feng_m1$target_esi2 == "2")
idx_not_2 <- which(df_feng_m1$target_esi2 == "not_2")

kept_2     <- sample(idx_2,     size = round(length(idx_2)     * keep_ratio_2))
kept_not_2 <- sample(idx_not_2, size = round(length(idx_not_2) * keep_ratio_not_2))

df_feng_m1 <- df_feng_m1[sort(c(kept_2, kept_not_2)), ]
df_feng_m1 <- na.omit(df_feng_m1)

cat(sprintf("MODEL 1 Dataset Ready (ESI 1 & 5 Excluded | Ratios: ESI 2=%.0f%%, Not ESI 2=%.0f%%): %d rows x %d cols\n",
            keep_ratio_2 * 100, keep_ratio_not_2 * 100, nrow(df_feng_m1), ncol(df_feng_m1)))
cat("Model 1 Target Distribution (2 vs not_2):\n")
print(table(df_feng_m1$target_esi2))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3A: Partitioning & Scaling for Model 1 (ESI 2 vs Not 2)
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size  <- config$training$val_size

in_train_val_m1 <- createDataPartition(df_feng_m1$target_esi2, p = 1 - test_size, list = FALSE)
train_val_m1    <- df_feng_m1[in_train_val_m1, ]
test_m1         <- df_feng_m1[-in_train_val_m1, ]

rel_val_size <- val_size / (1 - test_size)
in_train_m1  <- createDataPartition(train_val_m1$target_esi2, p = 1 - rel_val_size, list = FALSE)
train_m1     <- train_val_m1[in_train_m1, ]
val_m1       <- train_val_m1[-in_train_m1, ]

preproc_m1 <- preProcess(train_m1[, "age", drop = FALSE], method = c("center", "scale"))
train_m1   <- predict(preproc_m1, train_m1)
val_m1     <- predict(preproc_m1, val_m1)
test_m1    <- predict(preproc_m1, test_m1)

cat(sprintf("Model 1 Splits:\n  Train: %d rows | Val: %d rows | Test: %d rows\n", nrow(train_m1), nrow(val_m1), nrow(test_m1)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4A: Train Model 1 (Binary Logistic Regressor for ESI 2 vs Not 2)
# ---------------------------------------------------------
set.seed(config$training$random_state)

feat_names_m1 <- setdiff(names(train_m1), "target_esi2")
formula_m1 <- as.formula(paste("target_esi2 ~", paste(feat_names_m1, collapse = " + ")))

cat("Training Model 1 (ESI 2 vs Not ESI 2)...\n")
lr_esi2_model <- multinom(formula_m1, data = train_m1, trace = FALSE, MaxNWts = 5000)

cat("Model 1 (ESI 2) training complete!\n")
print(summary(lr_esi2_model))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5A: Comprehensive Evaluation of Model 1 (ESI 2 vs Not ESI 2)
# ---------------------------------------------------------
res_train_m1 <- evaluate_binary_lr(lr_esi2_model, train_m1, "target_esi2", "2", "MODEL 1 (ESI 2 vs Not 2)", "Train")
res_val_m1   <- evaluate_binary_lr(lr_esi2_model, val_m1,   "target_esi2", "2", "MODEL 1 (ESI 2 vs Not 2)", "Validation")
res_test_m1  <- evaluate_binary_lr(lr_esi2_model, test_m1,  "target_esi2", "2", "MODEL 1 (ESI 2 vs Not 2)", "Test")

metrics_m1 <- data.frame(
  Split     = factor(c("Train", "Validation", "Test"), levels = c("Train", "Validation", "Test")),
  Accuracy  = c(res_train_m1$acc,  res_val_m1$acc,  res_test_m1$acc),
  Precision = c(res_train_m1$prec, res_val_m1$prec, res_test_m1$prec),
  Recall    = c(res_train_m1$rec,  res_val_m1$rec,  res_test_m1$rec),
  PR_AUC    = c(res_train_m1$pr_auc, res_val_m1$pr_auc, res_test_m1$pr_auc),
  Log_Loss  = c(res_train_m1$log_loss, res_val_m1$log_loss, res_test_m1$log_loss)
)

cat("=== MODEL 1 (ESI 2 vs Not 2) OVERALL METRICS COMPARISON ===\n")
print(metrics_m1)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6A: Save Model 1 Diagnostic Visualizations
# ---------------------------------------------------------
if (!dir.exists("../plots")) dir.create("../plots", recursive = TRUE)

metrics_m1_long <- metrics_m1 %>%
  pivot_longer(cols = c("Accuracy", "Precision", "Recall", "PR_AUC"), names_to = "Metric", values_to = "Score")

p_bar_m1 <- ggplot(metrics_m1_long, aes(x = Metric, y = Score, fill = Split)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.7), width = 0.6) +
  geom_text(aes(label = sprintf("%.3f", Score)), position = position_dodge(width = 0.7), vjust = -0.3, size = 3) +
  theme_minimal() +
  scale_fill_manual(values = c("Train" = "#2b5c8f", "Validation" = "#e07a5f", "Test" = "#81b29a")) +
  labs(title = "Model 1 (ESI 2 vs Not 2) Metrics Comparison Across Splits",
       y = "Metric Value", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13), legend.position = "top")

ggsave("../plots/lr_esi2_metrics_barchart.png", plot = p_bar_m1, width = 9, height = 5, dpi = 300)
cat("Model 1 Bar Chart saved to: plots/lr_esi2_metrics_barchart.png\n")

p_bar_m1

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7A: Save Model 1 (ESI 2 vs Not 2) Artifact
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)

model1_path <- file.path(deploy_dir, "lr_feng_esi2_model.rds")
saveRDS(list(model = lr_esi2_model, preproc = preproc_m1), file = model1_path)
cat("Model 1 (ESI 2 vs Not 2) saved to:", model1_path, "\n")

--- 
## SECTION B: MODEL 2 - BINARY LOGISTIC REGRESSOR FOR ESI 3 vs. ESI 4
**Dataset Condition**: ESI 1, ESI 5, AND ESI 2 rows are completely excluded (`raw_esi %in% c("3", "4")`).

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2B: Load Data, Exclude ESI 1, 5, AND 2, Construct Target & Apply Class Subsampling
# ---------------------------------------------------------
set.seed(config$training$random_state)

df_feng_m2 <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)

raw_esi_m2 <- as.character(raw_df[[target_col]])

# EXCLUDE ESI 1, ESI 5, AND ESI 2 COMPLETELY FOR MODEL 2
idx_m2 <- which(raw_esi_m2 %in% c("3", "4") & !is.na(raw_esi_m2))
df_feng_m2  <- df_feng_m2[idx_m2, ]
raw_esi_sub2 <- raw_esi_m2[idx_m2]

# Target: '3' vs '4'
df_feng_m2$target_esi34 <- factor(raw_esi_sub2, levels = c("3", "4"))

# ---------------------------------------------------------
# Configurable Subsampling Ratios for Model 2
# ---------------------------------------------------------
keep_ratio_3 <- 0.5  # Keep 100% of ESI 3 rows (configurable)
keep_ratio_4 <- 0.4  # Keep 100% of ESI 4 rows (configurable)

idx_3 <- which(df_feng_m2$target_esi34 == "3")
idx_4 <- which(df_feng_m2$target_esi34 == "4")

kept_3 <- sample(idx_3, size = round(length(idx_3) * keep_ratio_3))
kept_4 <- sample(idx_4, size = round(length(idx_4) * keep_ratio_4))

df_feng_m2 <- df_feng_m2[sort(c(kept_3, kept_4)), ]
df_feng_m2 <- na.omit(df_feng_m2)

cat(sprintf("MODEL 2 Dataset Ready (ESI 1, 5 & 2 Excluded | Ratios: ESI 3=%.0f%%, ESI 4=%.0f%%): %d rows x %d cols\n",
            keep_ratio_3 * 100, keep_ratio_4 * 100, nrow(df_feng_m2), ncol(df_feng_m2)))
cat("Model 2 Target Distribution (3 vs 4):\n")
print(table(df_feng_m2$target_esi34))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3B: Partitioning & Scaling for Model 2 (ESI 3 vs ESI 4)
# ---------------------------------------------------------
set.seed(config$training$random_state)

in_train_val_m2 <- createDataPartition(df_feng_m2$target_esi34, p = 1 - test_size, list = FALSE)
train_val_m2    <- df_feng_m2[in_train_val_m2, ]
test_m2         <- df_feng_m2[-in_train_val_m2, ]

in_train_m2  <- createDataPartition(train_val_m2$target_esi34, p = 1 - rel_val_size, list = FALSE)
train_m2     <- train_val_m2[in_train_m2, ]
val_m2       <- train_val_m2[-in_train_m2, ]

preproc_m2 <- preProcess(train_m2[, "age", drop = FALSE], method = c("center", "scale"))
train_m2   <- predict(preproc_m2, train_m2)
val_m2     <- predict(preproc_m2, val_m2)
test_m2    <- predict(preproc_m2, test_m2)

cat(sprintf("Model 2 Splits:\n  Train: %d rows | Val: %d rows | Test: %d rows\n", nrow(train_m2), nrow(val_m2), nrow(test_m2)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4B: Train Model 2 (Binary Logistic Regressor for ESI 3 vs ESI 4)
# ---------------------------------------------------------
set.seed(config$training$random_state)

feat_names_m2 <- setdiff(names(train_m2), "target_esi34")
formula_m2 <- as.formula(paste("target_esi34 ~", paste(feat_names_m2, collapse = " + ")))

cat("Training Model 2 (ESI 3 vs ESI 4)...\n")
lr_esi34_model <- multinom(formula_m2, data = train_m2, trace = FALSE, MaxNWts = 5000)

cat("Model 2 (ESI 3 vs 4) training complete!\n")
print(summary(lr_esi34_model))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5B: Comprehensive Evaluation of Model 2 (ESI 3 vs ESI 4)
# ---------------------------------------------------------
res_train_m2 <- evaluate_binary_lr(lr_esi34_model, train_m2, "target_esi34", "3", "MODEL 2 (ESI 3 vs 4)", "Train")
res_val_m2   <- evaluate_binary_lr(lr_esi34_model, val_m2,   "target_esi34", "3", "MODEL 2 (ESI 3 vs 4)", "Validation")
res_test_m2  <- evaluate_binary_lr(lr_esi34_model, test_m2,  "target_esi34", "3", "MODEL 2 (ESI 3 vs 4)", "Test")

metrics_m2 <- data.frame(
  Split     = factor(c("Train", "Validation", "Test"), levels = c("Train", "Validation", "Test")),
  Accuracy  = c(res_train_m2$acc,  res_val_m2$acc,  res_test_m2$acc),
  Precision = c(res_train_m2$prec, res_val_m2$prec, res_test_m2$prec),
  Recall    = c(res_train_m2$rec,  res_val_m2$rec,  res_test_m2$rec),
  PR_AUC    = c(res_train_m2$pr_auc, res_val_m2$pr_auc, res_test_m2$pr_auc),
  Log_Loss  = c(res_train_m2$log_loss, res_val_m2$log_loss, res_test_m2$log_loss)
)

cat("=== MODEL 2 (ESI 3 vs 4) OVERALL METRICS COMPARISON ===\n")
print(metrics_m2)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6B: Save Model 2 Diagnostic Visualizations
# ---------------------------------------------------------
metrics_m2_long <- metrics_m2 %>%
  pivot_longer(cols = c("Accuracy", "Precision", "Recall", "PR_AUC"), names_to = "Metric", values_to = "Score")

p_bar_m2 <- ggplot(metrics_m2_long, aes(x = Metric, y = Score, fill = Split)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.7), width = 0.6) +
  geom_text(aes(label = sprintf("%.3f", Score)), position = position_dodge(width = 0.7), vjust = -0.3, size = 3) +
  theme_minimal() +
  scale_fill_manual(values = c("Train" = "#2b5c8f", "Validation" = "#e07a5f", "Test" = "#81b29a")) +
  labs(title = "Model 1 (ESI 3 vs ESI 4) Metrics Comparison Across Splits",
       y = "Metric Value", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13), legend.position = "top")

ggsave("../plots/lr_esi34_metrics_barchart.png", plot = p_bar_m2, width = 9, height = 5, dpi = 300)
cat("Model 2 Bar Chart saved to: plots/lr_esi34_metrics_barchart.png\n")

p_bar_m2

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7B: Save Model 2 (ESI 3 vs ESI 4) Artifact
# ---------------------------------------------------------
model2_path <- file.path(deploy_dir, "lr_feng_esi34_model.rds")
saveRDS(list(model = lr_esi34_model, preproc = preproc_m2), file = model2_path)
cat("Model 2 (ESI 3 vs 4) saved to:", model2_path, "\n")